Result 1: Table 1 Reproduction

Compares Degraded/Direct/Algorithm 1/Algorithm 2 reconstructions on MNIST / CIFAR-10/ CelebA, scoring each on FID/ SSIM/ RMSE

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("../code").resolve()))
sys.path.insert(0, str(pathlib.Path("..").resolve()))

import torch
from data.datasets import get_loader
from model import build_unet
from degradation import Degradation
from evaluate import evaluate_model

device= "cuda" if torch.cuda.is_available() else "cpu"

print("device: ", device)

In [ ]:
DATASETS = {
    "mnist": dict(
        kernel_size= 11, num_timesteps=20, kernel_std= 7.0, 
        blur_routine="Constant", image_channels=1,
    ),
    "cifar10": dict(
        kernel_size=11, num_timesteps=50, kernel_std=0.1, 
        blur_routine= "Special6", image_channels=3,
    ),
    "celeba": dict(
        kernel_size=15, num_timesteps=200, kernel_std= 0.01, 
        blur_routine="Exponential_Reflect", image_channels= 3,
    ),

}

CHECKPOINTS= {
    "mnist": "../checkpoints/mnist.pt",
    "cifar10": "../checkpoints/cifar10.pt",
    "celeba": "../checkpoints/celeba.pt",

}

BATCH_SIZE= 64

In [ ]:
def run_one(dataset_name):
    cfg= DATASETS[dataset_name]
    T = cfg["num_timesteps"]

    test_loader= get_loader(dataset_name, "test", batch_size= BATCH_SIZE, num_workers= 2, shuffle= False)
    degradation= Degradation(**cfg).to(device)
    model = build_unet(dataset_name).to(device)
    ckpt= torch.load(CHECKPOINTS[dataset_name], map_location=device)
    # use ema weights and not raw model
    model.load_state_dict(ckpt["ema"])
    model.eval()

    return evaluate_model(model, degradation, test_loader, t=T, device= device)




In [ ]:
results= {}

for i in DATASETS: 
    print(f"--- {i} ---")
    results[i] = run_one(i)
    print(results[i])



In [ ]:
def format_table(results):
    out = ["| Dataset | Method  | FID  | SSIM  | RMSE  |",
           "|---------|---------|------|-------|--------|"]
    for ds, res in results.items(): 
        for method in ["degraded", "direct", "alg1", "alg2"]:
            r= res[method]
            out.append(f"| {ds:<7} | {method:<8} | {r['fid']:6.2f} | {r['ssim']:.4f} | {r['rmse']:.4f} |")
    
    return "\n".join(out)

table= format_table(results)

print(table)

In [ ]:
import json, os

os.makedirs("../results", exist_ok= True)

with open("../results/table1.json", "w") as f:
    json.dump(results, f, indent= 2)

with open("../results/table1.md", "w") as f:
    f.write(table)

print("saved to ../results/")